# 04 — End-to-End QA & Release Gate Validation (`Agent 11` / `Agent 09`)

- **Active Run ID**: `run_20261111_002`
- **Orchestration & QA Gate Owner**: Agent 11 (`src/pipeline/orchestrator.py`, `src/hardcoding_scanner.py`)
- **Independent Review Consumer**: Agent 09 (`src/review/`)

## Responsibilities
1. Run Gate 1 (`verify_gate_1`) and zero-hardcoding verification.
2. Run all 9 automated `FreshnessChecker` repository memory/provenance checks.
3. Evaluate the evidence-derived `PipelineOrchestrator` state machine and 13-condition **Agent 03 Hard Release Gate** (`READY` vs `BLOCKED`).

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/HFG'), Path('/content')]
    for p in candidates:
        if (p / 'src').is_dir() and ((p / 'pyproject.toml').is_file() or (p / 'data').is_dir()):
            return p.resolve()
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None and Path('/content').is_dir():
    target = Path('/content/HFG')
    print(f'Repository not found locally; cloning into {target}...')
    subprocess.run(['git', 'clone', 'https://github.com/necantis/HFG.git', str(target)], check=True)
    REPO_ROOT = _find_repo_root()

if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate HFG repo root from {Path.cwd()}.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'HFG repo root: {REPO_ROOT}')

from src.hardcoding_scanner import HardcodingScanner, verify_gate_1
from src.ingest.ingestion import ACTIVE_RUN_ID

gate1 = verify_gate_1(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID)
scanner = HardcodingScanner(base_dir=REPO_ROOT)
hardcoding_hits = scanner.scan_codebase(['src', 'notebooks'])
print('Gate 1 status:', gate1['status'], '| Hardcoding hits:', len(hardcoding_hits))


In [ ]:
from src.freshness_checker import FreshnessChecker

freshness = FreshnessChecker(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID).run_all_checks()
print(f'Freshness passed categories: {len(freshness.passed_categories)}/9 | issues: {freshness.issues_found}')
for issue in freshness.issues:
    print(f'  [{issue.severity}] {issue.category} -> {issue.target}: {issue.description}')


In [ ]:
from src.pipeline.orchestrator import PipelineOrchestrator

write_log = os.environ.get('HFG_WRITE_EXECUTION_LOG') == '1'
report = PipelineOrchestrator(base_dir=REPO_ROOT, run_id=ACTIVE_RUN_ID).evaluate(write_log=write_log)

print('=== Pipeline State Machine ===')
print(json.dumps(report.states, indent=2))
print('\n=== Agent 03 Hard Release Gate Checklist ===')
print(json.dumps(report.agent_03_release_checklist, indent=2))
print(f'\nAGENT_03_RELEASE = {report.agent_03_release_decision.value}')
if report.blocking_reasons:
    print('Blocking reasons:')
    for reason in report.blocking_reasons:
        print(f'  - {reason}')
